# 07 Feedback loop: teacher quality and noise dose

Notebooks 05 and 06 use teachers trained on fewer rows than the production model, so their
pseudo-labels are noisier than the real derived labels. This notebook separates the two effects:

**Part 1: how noisy are the real derived labels?** Teachers are trained on growing shares of the
organic training folds. Their error among accepted predictions is measured on the clean test fold
(*organic*), and transferred to the pool through a calibration map, i.e. accuracy as a function of
confidence (*pool*). Both are fitted against training size and extrapolated to the production model's
size. When notebook 04's confidences are available, the calibration map is also applied directly to the
production confidences of the derived rows (*direct*).

**Part 2: how much does that noise cost?** Organic rows outside the test fold are split into a base
and a simulated pool, sized so that pool / base equals the production ratio after acceptance. A teacher
trained on the base labels the simulated pool. Because the true labels are known, a nested share
(the *dose*) of the teacher's errors is kept and the rest are replaced by the truth, which varies the noise
from 0 up to the teacher's own. The gain at the Part 1 noise estimate is the predicted production gain.

Caveat for Part 2: the simulated pool comes from the organic population, not from the items the model
actually predicted. It tests the ratio and the noise level, not the shift in distribution.

Checkpointed in `outputs/feedback_noise_dose_<timestamp>/`; set `RESUME_FROM` to continue.

In [ ]:
# ---- MLflow -------------------------------------------------------------------
USE_MLFLOW = False                    # False = no tracking at all
MLFLOW_EXPERIMENT = None              # None = this notebook's experiment; or e.g. "/Users/patrick.white@vallen.com/nb-classifier"

# ---- experiment -----------------------------------------------------------------
N_FOLDS = 3                           # ID-hash folds of the organic rows (test folds)
FOLD_SEED = 0
FOLDS_TO_RUN = None                   # None = all; e.g. [0] for a quick first look
TEACHER_FRACTIONS = [0.1, 0.2, 0.35, 0.5, 0.75, 1.0]  # part 1: teacher training sizes (shares of the training folds)
NOISE_DOSES = [0.0, 0.25, 0.5, 0.75, 1.0]             # part 2: share of the teacher's errors kept (0 = true labels)
ACCEPT_RATE = None                    # part 2: share of the pool that passes acceptance; None = from the data
SAMPLE_SEED = 0                       # seed of the nested subsamples and splits
PRODUCTION_RATIO = None               # None = derived rows / organic rows in the data
DEPTH = None                          # level the analysis uses (None = deepest, i.e. the full path)
CALIBRATION_BUCKET = 0.05             # width of the confidence buckets of the calibration map
N_BOOT = 300                          # paired bootstrap replicates over test rows
CI = 0.90                             # width of the intervals
RESUME_FROM = None                    # e.g. "feedback_noise_dose_20261020_143000"

In [ ]:
# ---- prepared data ------------------------------------------------------------
PREPARED_DATA = None                  # a data/<timestamp> folder from notebook 01 ("latest" = newest);
                                      # None = pull and clean below with the settings in this cell

# ---- data source (RAW_DATA_FILE supersedes SQL_QUERY) -----------------------
RAW_DATA_FILE = None                  # e.g. "data/my_extract.txt"; read with dtype=str
RAW_DATA_DELIMITER = "|"              # delimiter of RAW_DATA_FILE
RAW_DATA_QUOTED = True                # False if RAW_DATA_FILE uses no quoting and descriptions may start with "
SQL_QUERY = "labeled_items"           # a file in data/queries/ (name without .sql); may be None only if RAW_DATA_FILE is set

# ---- items to exclude ---------------------------------------------------------
BLACKLIST_FILE = None                 # e.g. "data/blacklists/blacklist.txt" (one column)
BLACKLIST_COL = "Vallen ID"           # data column the blacklist values are matched against
BLACKLIST_HAS_HEADER = True           # first line of the blacklist file is a header

# ---- cleaning -----------------------------------------------------------------
CLEANING_METHOD = "cleaning_method_20261015"   # a folder in cleaning/ (code, term lists, settings in method.yaml)
APPEND_FIELDS = {}                    # e.g. {"Global Vendno": "VEND", "Prodline": "PLINE"} -> VEND_<value> tokens
APPEND_NULLS_AS_NONE = True           # True: null field -> VEND_None;  False: no marker for that row

# ---- columns ------------------------------------------------------------------
TEXT_COL = "FullDesc"                 # raw description field to clean
CLEANED_COL = "CleanDesc"             # header for the cleaned description
LEVEL_COLUMNS = ["Level 1 ID", "Level 2 ID", "Level 3 ID", "Level 4 ID"]      # labels the model learns
ID_COL = "ID"                         # item key: matched against the prediction files, hashed for folds and subsamples

# ---- which labelled rows came from predictions ----------------------------------
PREDICTION_FILES = "all"              # notebook-04 outputs the derived labels came from: "all" (every
                                      # outputs/predictions_*), a folder name, a CSV path, or a list of these.
                                      # A labelled row whose ID is in these files counts as derived.
DERIVED_FLAG_COL = None               # or a data column marking derived rows (1/true/yes); overrides the ID match
POOL_SOURCE = "prediction_files"      # "prediction_files": every predicted item (accepted or not) is re-labelled
                                      #   by the fold teachers (needs INCLUDE_TEXT = True in notebook 04);
                                      # "derived": only the derived rows
ACCEPT_LEVEL = 4                      # predictions were accepted on this level's confidence ...
ACCEPT_MIN_CONFIDENCE = None          # ... when it was at least this; None = every prediction was accepted.
                                      # The diagnostics table below shows the threshold that was actually in use.

In [ ]:
# ==== HYPERPARAMETERS ==========================================================
# Single values only: the production model's settings (as in notebook 03). Every teacher and
# student in this notebook is trained with them.

HYPERPARAMETERS = {
    "vectorizer_kind":     "tfidf",
    "max_features":        None,
    "ngram_range":         (1, 2),
    "min_df":              2,
    "max_df":              0.95,
    "stop_words":          "english",
    "sublinear_tf":        True,
    "binary":              True,
    "norm":                "l2",
    "use_idf":             False,
    "ordered_bigrams":     True,
    "token_pattern":       "whitespace",
    "alpha":               0.001,
    "fit_prior":           True,
    "prediction_mode":     "conditional",
    "rare_path_threshold": 5,
}

In [ ]:
# ---- setup (no settings below this line need changing) ----------------------
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from nbclassifier import config, data, feedback, paths
from nbclassifier.tracking import Tracker

KIND = "noise_dose"
hp = config.expand_grid(HYPERPARAMETERS, grid_search=False)[0]
rule = feedback.SelectionRule(level=ACCEPT_LEVEL, min_confidence=ACCEPT_MIN_CONFIDENCE)
CONFIG = {
    "hyperparameters": hp.to_dict(), "n_folds": N_FOLDS, "fold_seed": FOLD_SEED, "sample_seed": SAMPLE_SEED,
    "rule": rule.to_dict(),
}
tracker = Tracker(USE_MLFLOW, MLFLOW_EXPERIMENT, run_name=f"07_feedback_noise_dose_{paths.timestamp()}",
                  tags={"notebook": "07_feedback_noise_dose"})

## Data

Organic rows are split from derived rows, and the pool (everything the model predicted) is built.
Check the diagnostics table:

* **recorded label = production prediction**: 1.0 means the derived labels are pure model output.
  Less than 1.0 means some were corrected after prediction, so they are partly validated. The
  experiments never use the recorded labels: the fold teachers re-label the pool, because the production
  model has seen every test fold. That is exact when the recorded labels are the model's predictions;
  the more were corrected, the more these experiments understate the real derived labels' quality.
* **production confidence of accepted vs not-accepted pool items**: if the two ranges don't
  overlap, the minimum of the accepted ones is the acceptance threshold that was in use. Set
  `ACCEPT_MIN_CONFIDENCE` to it so the fold teachers accept items the same way.

In [ ]:
if RESUME_FROM:
    store = feedback.Store.open(RESUME_FROM, KIND, CONFIG)
    fd = store.data()
    cols = feedback.Cols(**store.config["columns"])
    print(f"Resuming {store.folder.name}: the data saved in that folder is used; data settings above are ignored.")
else:
    if PREPARED_DATA:
        prepared = data.load_dataset(PREPARED_DATA)
        print(f"loaded {prepared.metadata['loaded_from']} (cleaning method {prepared.cleaner.method_name!r})")
        prepared = data.blacklist_prepared(prepared, BLACKLIST_FILE, BLACKLIST_COL, BLACKLIST_HAS_HEADER)
        TEXT_COL, CLEANED_COL = prepared.metadata["text_col"], prepared.metadata["cleaned_col"]
    else:
        prepared = data.prepare_dataset(
            raw_data_file=RAW_DATA_FILE, sql_query=SQL_QUERY, delimiter=RAW_DATA_DELIMITER,
            blacklist_file=BLACKLIST_FILE, blacklist_col=BLACKLIST_COL, blacklist_has_header=BLACKLIST_HAS_HEADER,
            cleaning_method=CLEANING_METHOD,
            append_fields=APPEND_FIELDS, append_nulls_as_none=APPEND_NULLS_AS_NONE,
            text_col=TEXT_COL, cleaned_col=CLEANED_COL, level_columns=LEVEL_COLUMNS,
            raw_data_quoted=RAW_DATA_QUOTED, spark=globals().get("spark"),
        )
    fd = feedback.split_feedback(prepared, level_columns=LEVEL_COLUMNS, text_col=TEXT_COL, cleaned_col=CLEANED_COL,
                                 id_col=ID_COL, prediction_files=PREDICTION_FILES,
                                 derived_flag_col=DERIVED_FLAG_COL, pool_source=POOL_SOURCE)
    del prepared
    cols = feedback.Cols(id=ID_COL, cleaned=CLEANED_COL, levels=list(LEVEL_COLUMNS))
    store = feedback.Store.create(KIND, {**CONFIG, "columns": cols.__dict__}, fd)
LEVEL_COLUMNS = cols.levels
DEPTH = DEPTH or len(LEVEL_COLUMNS)
N_PROD = len(fd.organic)                       # organic rows the production model is trained on
R_PROD = PRODUCTION_RATIO if PRODUCTION_RATIO is not None else fd.production_ratio
print(f"\nproduction: {N_PROD:,} organic rows, ratio {R_PROD:.3f}; analysis at L{DEPTH}")
print("evaluation folder:", store.folder)
feedback.describe(fd, LEVEL_COLUMNS, rule)

## Part 1: teacher error vs teacher size

One teacher per (fold, size). Its predictions on the test fold give the error and calibration; its
confidences on the pool give the acceptance share and the confidence mix of the accepted items.

In [ ]:
results = feedback.run_teacher_quality(
    store, fd, hp, cols, n_folds=N_FOLDS, fold_seed=FOLD_SEED, folds_to_run=FOLDS_TO_RUN,
    teacher_fracs=TEACHER_FRACTIONS, rule=rule, sample_seed=SAMPLE_SEED,
)

In [ ]:
meta = store.meta()
tq = meta[(meta.get("table") == "teacher") & meta["teacher_frac"].isin(TEACHER_FRACTIONS)].copy()
fracs = sorted(tq["teacher_frac"].unique())
folds_tq = sorted(tq.groupby("fold")["teacher_frac"].nunique().loc[lambda s: s == len(fracs)].index)
tq = tq[tq["fold"].isin(folds_tq)]

def keyof(prefix, k, g):
    return (feedback.teacher_key if prefix == "teacher" else feedback.pool_key)(k, g)

conf_col = f"L{ACCEPT_LEVEL} Confidence"
rows, cmaps = [], {}
for g in fracs:
    tt = pd.concat([store.table(keyof("teacher", k, g)) for k in folds_tq], ignore_index=True)
    pp = pd.concat([store.table(keyof("pool", k, g)) for k in folds_tq], ignore_index=True)
    acc_t = rule.mask(tt)
    cmap = feedback.calibration_map(tt.loc[acc_t, conf_col], tt.loc[acc_t, f"L{DEPTH} Correct"], CALIBRATION_BUCKET)
    cmaps[g] = cmap
    pool_acc = pp["accepted"].to_numpy()
    eta_pool = float(1 - feedback.apply_calibration(cmap, pp.loc[pool_acc, conf_col]).mean()) if pool_acc.any() else np.nan
    rows.append({"teacher_frac": g, "n_organic": tq.loc[tq["teacher_frac"] == g, "n_organic"].mean(),
                 f"teacher L{DEPTH} accuracy": float(tt[f"L{DEPTH} Correct"].mean()),
                 "test accept share": float(acc_t.mean()),
                 "pool accept share": float(pool_acc.mean()),
                 "error among accepted (organic)": float(1 - tt.loc[acc_t, f"L{DEPTH} Correct"].mean()),
                 "error among accepted (pool, calibrated)": eta_pool})
quality = pd.DataFrame(rows)
quality.round(4)

### Extrapolation to the production teacher

Each error rate is fitted as `c + a * n^(-b)`, falling to a floor *c* as the training size *n* grows,
and read at `N_PROD`. If there are too few sizes for a power law, a log-linear fit is used. Per-fold fits
show the spread.

The *direct* estimate applies the largest teacher's calibration map to the confidences that the
production model gave the derived rows. It needs no extrapolation in size, but it assumes the
production model is as well calibrated as a teacher with `(N_FOLDS - 1) / N_FOLDS` of its rows.

In [ ]:
estimates = {}
for col in ["error among accepted (organic)", "error among accepted (pool, calibrated)"]:
    f = feedback.fit_power_law(quality["n_organic"], quality[col])
    per_fold = []
    for k in folds_tq:
        t_k = tq[tq["fold"] == k].sort_values("teacher_frac")
        if col.endswith("(organic)"):
            y_k = t_k[f"teacher L{DEPTH} error among accepted"].to_numpy()
        else:
            y_k = []
            for g in t_k["teacher_frac"]:
                pp = store.table(keyof("pool", k, g))
                a = pp["accepted"].to_numpy()
                y_k.append(float(1 - feedback.apply_calibration(cmaps[g], pp.loc[a, conf_col]).mean()) if a.any() else np.nan)
        per_fold.append(float(feedback.fit_power_law(t_k["n_organic"], y_k)["predict"](N_PROD)))
    estimates[col] = {"fit": f["form"], "at production": float(f["predict"](N_PROD)),
                      "per-fold min": min(per_fold), "per-fold max": max(per_fold),
                      "at largest teacher": float(quality[col].iloc[-1])}

prod_conf_col = f"{feedback.PROD}L{ACCEPT_LEVEL} Confidence"
if prod_conf_col in fd.derived.columns and fd.derived[prod_conf_col].notna().any():
    c = fd.derived[prod_conf_col].dropna()
    eta_direct = float(1 - feedback.apply_calibration(cmaps[max(fracs)], c).mean())
    estimates["direct (production confidences)"] = {"fit": "calibration map of the largest teacher",
                                                    "at production": eta_direct, "per-fold min": np.nan,
                                                    "per-fold max": np.nan, "at largest teacher": np.nan}
est = pd.DataFrame(estimates).T
display(est)

# the noise used in part 2: pool estimate (direct one if available), with the per-fold range
ETA_KEY = "direct (production confidences)" if "direct (production confidences)" in estimates else "error among accepted (pool, calibrated)"
ETA_PROD = estimates[ETA_KEY]["at production"]
ETA_RANGE = (np.nanmin([e["at production"] for e in estimates.values()] + [estimates["error among accepted (pool, calibrated)"]["per-fold min"]]),
             np.nanmax([e["at production"] for e in estimates.values()] + [estimates["error among accepted (pool, calibrated)"]["per-fold max"]]))
print(f"estimated path noise of the derived labels: {ETA_PROD:.4f} ({ETA_KEY}); "
      f"range across estimates {ETA_RANGE[0]:.4f} to {ETA_RANGE[1]:.4f}")
agree = feedback.describe(fd, LEVEL_COLUMNS, rule)
agree = agree[agree["check"].str.startswith(f"recorded label = production prediction (through L{DEPTH})")]
if len(agree) and agree["value"].iloc[0] < 1:
    print(f"note: {1 - agree['value'].iloc[0]:.1%} of derived labels differ from the production prediction "
          "(corrected after prediction), so the recorded labels are somewhat cleaner than this estimate.")
tracker.log_metrics({"eta_prod": ETA_PROD, "eta_low": ETA_RANGE[0], "eta_high": ETA_RANGE[1]})

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
nn = np.geomspace(quality["n_organic"].min() * 0.9, N_PROD * 1.15, 100)
for col, color in [("error among accepted (organic)", "steelblue"), ("error among accepted (pool, calibrated)", "firebrick")]:
    ax.plot(quality["n_organic"], quality[col], "o", color=color, label=col)
    f = feedback.fit_power_law(quality["n_organic"], quality[col])
    ax.plot(nn, f["predict"](nn), color=color, lw=1)
    ax.plot([N_PROD], [estimates[col]["at production"]], "D", color=color)
if "direct (production confidences)" in estimates:
    ax.plot([N_PROD], [estimates["direct (production confidences)"]["at production"]], "*", color="black",
            markersize=12, label="direct (production confidences)")
ax.axvline(N_PROD, color="gray", ls=":", lw=1)
ax.set_xscale("log")
ax.set_xlabel("teacher training rows (organic)")
ax.set_ylabel(f"L{DEPTH} error among accepted predictions")
ax.set_title("Pseudo-label noise vs teacher size")
ax.legend(fontsize=8)
fig.tight_layout()
plot_dir = store.folder / "plots"
plot_dir.mkdir(exist_ok=True)
fig.savefig(plot_dir / f"teacher_noise_L{DEPTH}.png", dpi=120, bbox_inches="tight")
tracker.log_figure(fig, f"plots/teacher_noise_L{DEPTH}.png")

## Part 2: noise dose at the production ratio

The base is about `1 / (1 + 1.25 * R_PROD / ACCEPT_RATE)` of the training folds; the simulated pool is
sized with a 25% margin and then capped at exactly `R_PROD x base`. The base is smaller than production
and its teacher is noisier, which is why the dose is varied rather than taken as given.

In [ ]:
if ACCEPT_RATE is None:
    if len(fd.pool) > len(fd.derived) and rule.min_confidence is not None:
        ACCEPT_RATE = len(fd.derived) / len(fd.pool)
        how = "derived / pool"
    elif rule.min_confidence is not None:
        ACCEPT_RATE = float(quality["test accept share"].iloc[-1])
        how = "largest teacher's acceptance share on the test fold"
    else:
        ACCEPT_RATE, how = 1.0, "no acceptance threshold"
    print(f"ACCEPT_RATE = {ACCEPT_RATE:.3f} ({how})")
R_TARGET = round(float(R_PROD), 4)
results = feedback.run_noise_dose(
    store, fd, hp, cols, n_folds=N_FOLDS, fold_seed=FOLD_SEED, folds_to_run=FOLDS_TO_RUN,
    ratio=R_TARGET, accept_rate=ACCEPT_RATE, doses=NOISE_DOSES, rule=rule, sample_seed=SAMPLE_SEED,
)

In [ ]:
meta = store.meta()
dm = meta[meta["key"].str.startswith("dose_")].copy()
dm["cell"] = dm["dose"].fillna(-1.0)
want = [-1.0] + sorted(NOISE_DOSES)
folds_d = sorted(dm[dm["cell"].isin(want)].groupby("fold")["cell"].nunique().loc[lambda s: s == len(want)].index)
if not folds_d:
    raise RuntimeError("No fold has finished every dose yet")
dm = dm[dm["fold"].isin(folds_d) & dm["cell"].isin(want)]
keys = [{int(r.fold): r.key for r in dm[dm["cell"] == c].itertuples()} for c in want]
point, boot = feedback.bootstrap_cells(store, keys, DEPTH, n_boot=N_BOOT)
q = [(1 - CI) / 2, 1 - (1 - CI) / 2]
d_point, d_boot = point[1:] - point[0], boot[:, 1:] - boot[:, [0]]
g = dm.groupby("cell")
dose = pd.DataFrame({
    "dose": sorted(NOISE_DOSES),
    "noise": g["noise"].mean().loc[sorted(NOISE_DOSES)].to_numpy(),
    "n_base": g["n_base"].mean().loc[sorted(NOISE_DOSES)].round().astype(int).to_numpy(),
    "n_sim_pool": g["n_derived"].mean().loc[sorted(NOISE_DOSES)].round().astype(int).to_numpy(),
    "ratio_realized": g["ratio"].mean().loc[sorted(NOISE_DOSES)].to_numpy(),
    "accuracy with": point[1:], "delta": d_point,
    "delta_low": np.quantile(d_boot, q[0], axis=0), "delta_high": np.quantile(d_boot, q[1], axis=0),
    "delta_se": d_boot.std(axis=0, ddof=1),
})
print(f"folds used: {folds_d}; accuracy without (base only): {point[0]:.4f}; "
      f"teacher path noise on the simulated pool: {dm['teacher_noise'].mean():.4f}")
dose.round(4)

### Gain as a function of noise, read at the estimated production noise

A straight line `delta = a + b * noise` is fitted (weighted, refitted on every bootstrap replicate).
*a* is the gain from the extra rows if their labels were perfect; *b* is the cost per unit of noise. The
interval at `ETA_PROD` covers the test-row noise only; the range from Part 1 is shown separately.

In [ ]:
x = dose["noise"].to_numpy()
w = 1.0 / np.maximum(dose["delta_se"].to_numpy(), 1e-6) ** 2
X = np.column_stack([np.ones_like(x), x])
sw = np.sqrt(w)[:, None]
coef = np.linalg.lstsq(X * sw, dose["delta"].to_numpy()[:, None] * sw, rcond=None)[0].ravel()
coef_boot = np.linalg.lstsq(X * sw, d_boot.T * sw, rcond=None)[0]          # 2 x N_BOOT
at = lambda e: coef[0] + coef[1] * e
at_boot = lambda e: coef_boot[0] + coef_boot[1] * e
lo, hi = np.quantile(at_boot(ETA_PROD), q)
inside = x.min() <= ETA_PROD <= x.max()
print(f"gain with perfect labels: {coef[0]:+.4f};  cost per 0.10 of noise: {coef[1] * 0.1:+.4f}")
if coef[0] > 0 and coef[1] < 0:
    print(f"break-even noise (gain = 0): {-coef[0] / coef[1]:.4f}")
elif coef[0] <= 0:
    print("even perfectly labelled extra rows of this kind don't help at this ratio (gain at noise 0 <= 0)")
print(f"\nat the estimated production noise {ETA_PROD:.4f} ({'inside' if inside else 'OUTSIDE'} the tested range "
      f"{x.min():.4f} to {x.max():.4f}):")
print(f"  predicted change in L{DEPTH} accuracy: {at(ETA_PROD):+.4f}  ({CI:.0%} interval {lo:+.4f} to {hi:+.4f})")
print(f"  across the Part 1 noise range {ETA_RANGE[0]:.4f} to {ETA_RANGE[1]:.4f}: "
      f"{at(ETA_RANGE[1]):+.4f} to {at(ETA_RANGE[0]):+.4f}")
print(f"  (the naive experiment, dose 1 with this smaller teacher: {dose['delta'].iloc[-1]:+.4f})")
if hi < 0:
    print("=> including the derived items is expected to LOWER accuracy")
elif lo > 0:
    print("=> including the derived items is expected to RAISE accuracy")
else:
    print("=> the interval includes zero: no reliable gain or loss")
tracker.log_metrics({"delta_perfect_labels": coef[0], "delta_per_noise": coef[1],
                     "delta_at_eta_prod": at(ETA_PROD), "delta_low": lo, "delta_high": hi})

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.errorbar(dose["noise"], dose["delta"], yerr=[dose["delta"] - dose["delta_low"], dose["delta_high"] - dose["delta"]],
            fmt="o", color="black", capsize=3, label="measured (dose 0 = true labels)")
xx = np.linspace(0, max(x.max(), ETA_RANGE[1]) * 1.05, 100)
band = coef_boot[0][:, None] + coef_boot[1][:, None] * xx[None, :]
ax.plot(xx, at(xx), color="firebrick", lw=1.2, label="linear fit")
ax.fill_between(xx, np.quantile(band, q[0], axis=0), np.quantile(band, q[1], axis=0), color="firebrick", alpha=0.15)
ax.axvspan(ETA_RANGE[0], ETA_RANGE[1], color="gold", alpha=0.25, label="Part 1 noise range")
ax.axvline(ETA_PROD, color="darkgoldenrod", ls="--", lw=1.2)
ax.errorbar([ETA_PROD], [at(ETA_PROD)], yerr=[[at(ETA_PROD) - lo], [hi - at(ETA_PROD)]], fmt="D",
            color="darkgoldenrod", capsize=4, label=f"estimated production noise: {at(ETA_PROD):+.4f}")
ax.axhline(0, color="gray", lw=0.8)
ax.set_xlabel(f"path noise of the added labels (share wrong at L{DEPTH})")
ax.set_ylabel(f"change in L{DEPTH} accuracy vs base only")
ax.set_title(f"Noise dose at ratio {R_TARGET}")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(plot_dir / f"noise_dose_L{DEPTH}.png", dpi=120, bbox_inches="tight")
tracker.log_figure(fig, f"plots/noise_dose_L{DEPTH}.png")

In [ ]:
quality.to_csv(store.folder / f"teacher_quality_L{DEPTH}.csv", index=False)
dose.to_csv(store.folder / f"noise_dose_L{DEPTH}.csv", index=False)
tracker.log_table(quality, f"teacher_quality_L{DEPTH}.csv")
tracker.log_table(dose, f"noise_dose_L{DEPTH}.csv")
tracker.end()